# Análisis Exploratorio de Datos - Analítica Educativa

Este notebook realiza un análisis exploratorio de los datos educativos para identificar patrones en deserción, rendimiento y demanda de asignaturas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.config import *
from src.data_loader import *
from src.preprocessor import *
from src.analyzer import *
from src.visualizer import *

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Carga de Datos

In [ ]:
# Cargar datos (descomentar cuando tengas los archivos CSV)
# df_estudiantes = cargar_datos_estudiantes('estudiantes.csv')
# df_academico = cargar_datos_academicos('rendimiento_academico.csv')

# Para demostración, crear datos sintéticos
np.random.seed(42)
n_estudiantes = 500
n_registros = 2000

df_estudiantes = pd.DataFrame({
    'id_estudiante': range(1, n_estudiantes + 1),
    'genero': np.random.choice(['M', 'F'], n_estudiantes),
    'edad': np.random.randint(16, 30, n_estudiantes),
    'estado_civil': np.random.choice(['Soltero', 'Casado', 'Divorciado'], n_estudiantes, p=[0.7, 0.25, 0.05]),
    'nacionalidad': np.random.choice(['Local', 'Extranjero'], n_estudiantes, p=[0.85, 0.15]),
    'modo_admision': np.random.choice(['Regular', 'Especial', 'Pasantía'], n_estudiantes, p=[0.7, 0.2, 0.1]),
    'becado': np.random.choice([True, False], n_estudiantes, p=[0.3, 0.7])
})

asignaturas = ['Matemáticas', 'Física', 'Química', 'Programación', 'Base de Datos', 'Redes']

df_academico = pd.DataFrame({
    'id_estudiante': np.random.choice(range(1, n_estudiantes + 1), n_registros),
    'semestre': np.random.choice(['2023-1', '2023-2', '2024-1', '2024-2'], n_registros),
    'ano_academico': np.random.choice([2023, 2024], n_registros),
    'id_asignatura': np.random.choice(range(1, len(asignaturas) + 1), n_registros),
    'nombre_asignatura': np.random.choice(asignaturas, n_registros),
    'creditos': np.random.choice([3, 4, 5], n_registros),
    'calificacion': np.random.normal(72, 15, n_registros).clip(0, 100).round(1),
    'aprobado': np.random.choice([True, False], n_registros, p=[0.7, 0.3]),
    'deserto': np.random.choice([True, False], n_registros, p=[0.15, 0.85])
})

print(f'Estudiantes: {len(df_estudiantes)}')
print(f'Registros académicos: {len(df_academico)}')

## 2. Calidad de Datos

In [ ]:
print('=== Calidad de datos - Estudiantes ===')
analizar_calidad_datos(df_estudiantes)

In [ ]:
print('=== Calidad de datos - Rendimiento Académico ===')
analizar_calidad_datos(df_academico)

## 3. Métricas Generales

In [ ]:
metricas = metricas_generales(df_estudiantes, df_academico)
print('=== Métricas Generales ===')
for k, v in metricas.items():
    print(f'{k}: {v}')

## 4. Análisis de Deserción

In [ ]:
# Merge para tener datos demográficos en análisis académico
df_completo = df_academico.merge(
    df_estudiantes[['id_estudiante', 'genero', 'edad', 'becado']],
    on='id_estudiante',
    how='left'
)

analisis_desercion = analizar_desercion(df_completo)

print('=== Deserción por Género ===')
display(analisis_desercion['por_genero'])

print('\n=== Deserción por Semestre ===')
display(analisis_desercion['por_semestre'])

In [ ]:
# Gráfica de deserción por género
fig = graficar_tasa_desercion(
    df_completo, 
    columna_grupo='genero',
    titulo='Tasa de Deserción por Género'
)
plt.show()

In [ ]:
print('=== Top 10 Asignaturas con Mayor Deserción ===')
display(analisis_desercion['por_asignatura'])

## 5. Análisis de Rendimiento

In [ ]:
analisis_rendimiento = analizar_rendimiento(df_academico)

print('=== Rendimiento por Asignatura ===')
display(analisis_rendimiento['por_asignatura'])

print('\n=== Distribución de Calificaciones ===')
display(analisis_rendimiento['distribucion'])

In [ ]:
# Gráfica de rendimiento por asignatura
fig = graficar_rendimiento_asignaturas(
    df_academico,
    top_n=6,
    titulo='Promedio por Asignatura'
)
plt.show()

In [ ]:
# Distribución de calificaciones
fig = graficar_distribucion_calificaciones(
    df_academico,
    titulo='Distribución General de Calificaciones'
)
plt.show()

## 6. Evolución Temporal

In [ ]:
fig = graficar_evolucion_temporal(
    df_academico,
    columna_fecha='semestre',
    columna_valor='calificacion',
    titulo='Evolución del Promedio por Semestre'
)
plt.show()

## 7. Correlaciones

In [ ]:
fig = graficar_mapa_calor_correlaciones(
    df_academico,
    columnas=['creditos', 'calificacion', 'aprobado', 'deserto'],
    titulo='Correlaciones en Datos Académicos'
)
plt.show()

## 8. Conclusiones y Recomendaciones

In [ ]:
metricas = metricas_generales(df_estudiantes, df_academico)
analisis_desercion = analizar_desercion(df_completo)

# Para demostración, crear datos de demanda sintéticos
df_demanda = pd.DataFrame({
    'nombre_asignatura': asignaturas,
    'demanda': np.random.randint(50, 200, len(asignaturas)),
    'inscritos': np.random.randint(40, 180, len(asignaturas)),
    'cupos_disponibles': np.random.randint(100, 200, len(asignaturas))
})

analisis_demanda = analizar_demanda_asignaturas(df_demanda)

recomendaciones = generar_recomendaciones(metricas, analisis_desercion, analisis_demanda)
print('=== Recomendaciones ===')
display(recomendaciones)